In [0]:
from pyspark.sql.functions import col, to_date,agg

## Load Bronze Tables

In [0]:
spark.sql("USE inventory_ai")

sales_bronze_df = spark.table("bronze_sales_raw")
inventory_bronze_df = spark.table("bronze_inventory_raw")
supplier_bronze_df = spark.table("bronze_suppliers_raw")


In [0]:
%sql
select * from bronze_sales_raw limit 5

Changing the date column to date format

In [0]:
from pyspark.sql.functions import col, to_date, sum as sum_

sales_clean_df = (
    sales_bronze_df
    .withColumn("date", to_date(col("date")))
)

sales_clean_df.show(5)
sales_clean_df.printSchema()


Renaming columns to business needs

In [0]:
sales_clean_df = (
    sales_clean_df
    .withColumnRenamed("store_nbr", "store_id")
    .withColumnRenamed("family", "product_family")
    .withColumnRenamed("sales", "daily_sales")
    .withColumnRenamed("onpromotion", "onpromotion_count")
)

sales_clean_df.printSchema()


In [0]:
sales_clean_df.printSchema()

In [0]:
sales_clean_df.select(
    sum_(col("daily_sales").isNull().cast("int")).alias("null_sales"),
    sum_(col("onpromotion_count").isNull().cast("int")).alias("null_promo")
).show()

In [0]:
sales_clean_df.filter(col("daily_sales") < 0).count()


In [0]:
sales_clean_df.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("inventory_ai.silver_daily_sales")


In [0]:
%sql
show tables

### Create Inventory Snapshot (Silver)

In [0]:
from pyspark.sql import Row

inventory_data = [
    Row(store_id=1, product_family="GROCERY I", current_stock=300),
    Row(store_id=1, product_family="BEVERAGES", current_stock=200),
    Row(store_id=1, product_family="CLEANING", current_stock=150),
    Row(store_id=2, product_family="GROCERY I", current_stock=250),
    Row(store_id=2, product_family="BEVERAGES", current_stock=180),
]

silver_inventory_df = spark.createDataFrame(inventory_data)

silver_inventory_df.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("inventory_ai.silver_inventory")


## Supplier lead time

In [0]:
supplier_data = [
    Row(product_family="GROCERY I", supplier_id="SUP_1", lead_time_days=5),
    Row(product_family="BEVERAGES", supplier_id="SUP_2", lead_time_days=7),
    Row(product_family="CLEANING", supplier_id="SUP_3", lead_time_days=10),
]

silver_supplier_df = spark.createDataFrame(supplier_data)

silver_supplier_df.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("inventory_ai.silver_supplier_lead_time")


In [0]:
spark.sql("""
SELECT *
FROM silver_daily_sales
ORDER BY date
LIMIT 10
""").display()
